# TP1 — EDA formal · Ejercicio 1

**73.69 Large Language Models — ITBA, 2026** · Predicción de *Buy Through Rate*

Este notebook es el entregable de las **Fases 1 y 2** del [plan](../plan.md): la auditoría
de fugas y el análisis exploratorio que responde los cuatro puntos del Ejercicio 1.

Todas las cifras se **miden acá**, no se copian del plan. La sección 1 compara lo medido
contra los valores que el plan declara, y falla ruidosamente si alguno no reproduce.

## Índice

| § | Contenido | Para qué |
|---|---|---|
| 0 | Auditoría de fugas | Fase 1: qué columnas pueden entrar al modelo |
| 1 | Verificación de los hechos del plan | Criterio de aceptación de la Fase 2 |
| 2 | Univariado de numéricas | Escalas y asimetría → preprocesamiento |
| 3 | Cardinalidad de categóricas | Decidir one-hot vs embedding |
| 4 | Estructura de queries | Justifica la partición agrupada y descarta *listwise* |
| 5 | Coherencia temporal | **Justifica descartar el split temporal (D4)** |
| 6 | Coherencia con los filtros | Justifica no construir features de match (D6) |
| 7 | Análisis del texto | **Donde está la señal**: el sufijo del título |
| 8 | Bivariado marginal con el target | Lo que se ve sin condicionar |
| 9 | Análisis condicionado al nivel ALTO | Lo que el análisis marginal esconde |
| 10 | Confusión Seafood / Fish-Shellfish | Dos nombres para la misma señal |
| 11 | Las cuatro respuestas del Ejercicio 1 | |

---
## Setup

In [ ]:
from __future__ import annotations

import re
import sys
from pathlib import Path

# El notebook corre desde notebooks/; la raiz del repo es el padre.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.data import load as L

FIGDIR = ROOT / "report" / "figures"
FIGDIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 150,
    "savefig.bbox": "tight",
    "font.size": 9,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

AZUL, GRIS, ROJO, VERDE = "#2b6cb0", "#a0aec0", "#c53030", "#2f855a"
COLOR_NIVEL = {"ALTO": ROJO, "MEDIO": "#dd6b20", "CERO": GRIS}


def guardar(fig, nombre):
    """Guarda la figura en report/figures/ para llevarla a la presentacion."""
    ruta = FIGDIR / f"{nombre}.png"
    fig.savefig(ruta)
    print(f"guardada: {ruta.relative_to(ROOT)}")


def tasa_por(df, col, target="bought"):
    """Tabla filas / compras / tasa por valor de una categorica."""
    return (df.groupby(col, observed=True)[target]
              .agg(filas="size", compras="sum", tasa="mean")
              .sort_values("tasa", ascending=False))

In [ ]:
df = L.load_raw()
print(f"{df.shape[0]:,} filas x {df.shape[1]} columnas")
print(f"rango de timestamp: {df['timestamp'].min():%Y-%m-%d} a {df['timestamp'].max():%Y-%m-%d}")
df.head(3)

---
## 0. Auditoría de fugas (Fase 1)

El leakage **no se detecta con validación cruzada**: si una columna posterior al momento de
la predicción entra al pipeline, las tres particiones quedan igualmente contaminadas y las
curvas se ven sanas. La única defensa es auditar cada columna contra un momento de
predicción definido por escrito, antes de entrenar nada.

In [ ]:
print(L.PREDICTION_MOMENT)

In [ ]:
# Perfil crudo: dtype, nulos, cardinalidad.
L.column_profile(df)

In [ ]:
# La tabla de auditoria: una fila por cada una de las 22 columnas.
auditoria = L.audit_table()
auditoria

In [ ]:
print(f"ADMITIDAS ({len(L.FEATURES_ADMITIDAS)}):")
for c in L.FEATURES_ADMITIDAS:
    print(f"   {c}")
print(f"\nEXCLUIDAS ({len(L.FEATURES_EXCLUIDAS)}):")
for c, motivo in L.FEATURES_EXCLUIDAS.items():
    print(f"   {c}: {motivo}")
print(f"\nTARGET: {L.TARGET}    GRUPO (no feature): {L.GROUP}")

### La fuga de `cart`

`cart` registra si el usuario agregó el producto al carrito. Su valor queda determinado
**después** de mostrar la página de resultados, es decir después del momento de la
predicción. Eso solo ya alcanza para excluirla. La tabla de contingencia muestra además
que la fuga es total.

In [ ]:
L.leakage_contingency(df)

`P(bought = 1 | cart = False) = 0.0000` sobre 6.993 filas, **cero exacto**. No es una
correlación alta: agregar al carrito es *condición necesaria* de la compra. Un modelo con
acceso a `cart` no predice, lee la respuesta parcial.

Vale la pena anticipar la pregunta de defensa: el baseline con `cart` da **peor** ROC-AUC
(0.9197) que el baseline con el sufijo del título (0.9695). La razón por la que `cart` se
excluye no es que infle la métrica, sino que **su valor no existe** en el instante en que
el sistema tiene que decidir qué promocionar.

---
## 1. Verificación de los hechos declarados en el plan

Criterio de aceptación de la Fase 2. Si alguna fila da `ok = False`, el CSV cambió y hay
que revisar el plan entero, no ajustar la tolerancia.

In [ ]:
verificacion = L.verify_expected_facts(df)
assert verificacion["ok"].all(), "hay hechos del plan que no se reproducen"
verificacion

---
## 2. Univariado de las numéricas

Interesa el **rango** y la **asimetría**, porque determinan el preprocesamiento: una
variable con cola larga a derecha se comporta mal con `StandardScaler` y pide
`QuantileTransformer` o log.

In [ ]:
NUMERICAS = ["price", "net_weight_oz", "nutrition_score", "filter_price_min", "filter_price_max"]

resumen = df[NUMERICAS].describe().T
resumen["skew"] = df[NUMERICAS].skew()
resumen["kurtosis"] = df[NUMERICAS].kurtosis()
resumen[["count", "mean", "std", "min", "50%", "max", "skew", "kurtosis"]]

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(15, 2.8))
for ax, col in zip(axes, NUMERICAS):
    ax.hist(df[col], bins=40, color=AZUL, edgecolor="white", linewidth=0.3)
    ax.set_title(f"{col}\nskew = {df[col].skew():.2f}", fontsize=9)
    ax.set_yticks([])
fig.suptitle("Distribución de las variables numéricas", y=1.08, fontsize=11)
fig.tight_layout()
guardar(fig, "fig_01_numericas")
plt.show()

`net_weight_oz` (skew 2.76), `filter_price_min` (1.67) y `price` (1.48) tienen cola larga a
derecha. `nutrition_score` es casi simétrica y se parece a una uniforme truncada, lo que ya
sugiere que es un atributo sintético sin relación con la conducta.

**Consecuencia para la Fase 4:** escalar con `QuantileTransformer` en lugar de
`StandardScaler`, o aplicar `log1p` a las tres asimétricas. En cualquier caso, el
transformador se ajusta **solo con train**.

---
## 3. Cardinalidad de las categóricas

Decide la codificación: one-hot para los baselines, `nn.Embedding` para el Transformer.

In [ ]:
CATEGORICAS = ["category", "brand", "storage_type", "country_of_origin",
               "unit_of_measure", "package_size", "ingredients", "allergens"]

pd.DataFrame([
    {
        "columna": c,
        "cardinalidad": df[c].nunique(dropna=True),
        "nulos %": round(df[c].isna().mean() * 100, 2),
        "valor mas frecuente": df[c].mode().iloc[0],
        "cobertura del top-1 %": round(df[c].value_counts(normalize=True).iloc[0] * 100, 1),
        "filas por valor (mediana)": int(df[c].value_counts().median()),
    }
    for c in CATEGORICAS
])

Todas son de cardinalidad baja o media. **No hace falta un bucket `[RARE]`**: incluso
`ingredients`, la más alta con 190 valores, tiene una mediana de 10 filas por valor.

`allergens` tiene 44.55% de nulos. El nulo **no** se imputa: es informativo.

In [ ]:
# El nulo de allergens tiene su propia tasa de compra, distinta de la de sus vecinos.
alergenos = df.assign(allergens=df["allergens"].fillna("[NONE]"))
tasa_por(alergenos, "allergens")

La tasa del nulo (0.1385) cae **entre** Milk y Soy, no en un extremo. Es una categoría más:
"producto sin alérgeno declarado". Imputarlo con la moda destruiría esa información y
metería 4.455 filas en una categoría a la que no pertenecen.

---
## 4. Estructura de queries

Cada `query_id` es una búsqueda con filtros; sus filas son las impresiones que devolvió.
Esto determina dos decisiones: cómo se particiona y cómo se formula el problema.

In [ ]:
items_por_query = df.groupby(L.GROUP).size()
compras_por_query = df.groupby(L.GROUP)["bought"].sum()

print(f"queries únicas      : {df[L.GROUP].nunique():,}")
print(f"ítems por query     : min {items_por_query.min()}, max {items_por_query.max()}, media {items_por_query.mean():.2f}")
print(f"queries sin compra  : {(compras_por_query == 0).sum():,} ({(compras_por_query == 0).mean():.1%})")
print()
print("distribución de compras por query:")
print(compras_por_query.value_counts().sort_index().to_string())

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3))

vc = items_por_query.value_counts().sort_index()
ax1.bar(vc.index, vc.values, color=AZUL)
ax1.set_title(f"Ítems por query (media {items_por_query.mean():.2f})")
ax1.set_xlabel("ítems"); ax1.set_ylabel("queries")

vc2 = compras_por_query.value_counts().sort_index()
barras = ax2.bar(vc2.index, vc2.values, color=[GRIS] + [ROJO] * (len(vc2) - 1))
ax2.set_title("Compras por query")
ax2.set_xlabel("compras"); ax2.set_ylabel("queries")
ax2.bar_label(barras, fontsize=8)

fig.tight_layout()
guardar(fig, "fig_02_estructura_queries")
plt.show()

**Dos lecturas.**

1. Las filas de una misma query comparten filtros y contexto. Un split aleatorio a nivel
   fila pondría impresiones de la misma búsqueda en train y en test, y el modelo podría
   memorizar el contexto en lugar de aprenderlo. → **partición agrupada por `query_id`**
   (decisión D3).

2. Hay **284 queries con más de una compra** (226 con 2, 53 con 3, 5 con 4). El problema
   **no es de elección única**, así que no corresponde una formulación *listwise* con
   softmax sobre la query. Se modela como **clasificación binaria a nivel impresión**
   (decisión D1), que además es la formulación que habilita PR-AUC y ROC-AUC.

---
## 5. Coherencia temporal — por qué NO hay split temporal

La intuición dice que en un problema de recomendación hay que partir por tiempo, para no
predecir el pasado con el futuro. Acá esa intuición **no aplica**, y conviene mostrar por
qué en lugar de afirmarlo.

In [ ]:
rango_intra = (df.groupby(L.GROUP)["timestamp"]
                 .agg(lambda s: (s.max() - s.min()).days)
                 .rename("dias"))

print(f"rango temporal DENTRO de una misma query (días):")
print(f"  mediana : {rango_intra.median():.0f}")
print(f"  media   : {rango_intra.mean():.0f}")
print(f"  máximo  : {rango_intra.max():.0f}")
print(f"  queries con rango > 1 año: {(rango_intra > 365).mean():.1%}")
print()
span_global = (df['timestamp'].max() - df['timestamp'].min()).days
print(f"rango temporal de TODO el dataset: {span_global} días")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.2))

ax1.hist(rango_intra, bins=40, color=AZUL, edgecolor="white", linewidth=0.3)
ax1.axvline(rango_intra.median(), color=ROJO, lw=2,
            label=f"mediana = {rango_intra.median():.0f} días")
ax1.axvline(span_global, color="black", lw=2, ls="--",
            label=f"rango del dataset = {span_global} días")
ax1.set_title("Rango temporal DENTRO de cada query")
ax1.set_xlabel("días entre la impresión más vieja y la más nueva de la query")
ax1.set_ylabel("queries")
ax1.legend(fontsize=8)

# Una muestra de queries, cada una como un segmento en el eje del tiempo.
muestra = df[L.GROUP].drop_duplicates().sample(40, random_state=42)
sub = df[df[L.GROUP].isin(muestra)]
for i, (_, g) in enumerate(sub.groupby(L.GROUP)):
    ax2.plot([g["timestamp"].min(), g["timestamp"].max()], [i, i],
             color=AZUL, lw=1.2, alpha=0.8, solid_capstyle="round")
    ax2.scatter(g["timestamp"], [i] * len(g), s=6, color=ROJO, zorder=3)
ax2.set_title("40 queries al azar: sus impresiones en el tiempo")
ax2.set_ylabel("query"); ax2.set_yticks([])
ax2.tick_params(axis="x", rotation=30)

fig.tight_layout()
guardar(fig, "fig_03_coherencia_temporal")
plt.show()

**Este gráfico justifica la decisión D4.**

El dataset abarca 730 días. El rango temporal *dentro de una misma query* tiene mediana de
**488 días** y máximo de 728: una sola búsqueda contiene impresiones separadas por más de
un año. El panel derecho lo muestra directo — cada segmento es una query, y casi todos
cruzan buena parte del eje.

El `timestamp` **no ordena las queries**: no existe un instante que separe "queries pasadas"
de "queries futuras". Un split temporal partiría cada query por la mitad, que es exactamente
la contaminación que la partición agrupada busca evitar. Se descarta.

---
## 6. Coherencia con los filtros

Un reflejo razonable sería construir features de "match": ¿la categoría del producto
coincide con la del filtro? ¿el precio cae dentro del rango pedido? Antes de construirlas,
hay que verificar que **varían**.

In [ ]:
L.filter_coherence(df)

Las tres condiciones se cumplen en el **100%** de las filas. Esas features serían columnas
constantes: cero varianza, cero información, y en el caso del one-hot, una columna
degenerada que solo agrega ruido de optimización. **No se construyen** (decisión D6).

Lo que sí varía es *dónde* cae el precio dentro del rango del filtro:

In [ ]:
df["price_pos"] = ((df["price"] - df["filter_price_min"])
                   / (df["filter_price_max"] - df["filter_price_min"]))

print(f"price_pos: min {df['price_pos'].min():.4f}  max {df['price_pos'].max():.4f}")
print(f"correlación con bought (marginal): {df['price_pos'].corr(df['bought'].astype(int)):.4f}")
print("\n(la correlación marginal es baja; en la §9 se ve que dentro del nivel ALTO sube a 0.105)")

---
## 7. Análisis del texto — donde está la señal

### 7.1 El sufijo del título

In [ ]:
df["sufijo"] = L.extract_suffix(df["title"])
tabla_sufijo = tasa_por(df, "sufijo")
tabla_sufijo

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5.5))
t = tabla_sufijo.sort_values("tasa")
colores = [ROJO if v > 0.4 else ("#dd6b20" if v > 0.005 else GRIS) for v in t["tasa"]]
barras = ax.barh(t.index, t["tasa"], color=colores)
ax.axvline(df["bought"].mean(), color="black", ls="--", lw=1.2,
           label=f"prevalencia global = {df['bought'].mean():.4f}")
ax.set_xlabel("tasa de compra")
ax.set_title("Tasa de compra por sufijo del título")
ax.bar_label(barras, fmt="%.3f", fontsize=7.5, padding=2)
ax.set_xlim(0, 0.78)
ax.legend(fontsize=8, loc="lower right")
guardar(fig, "fig_04_tasa_por_sufijo")
plt.show()

Los 20 valores (incluido *sin sufijo*) se separan en **tres niveles sin solapamiento**. No
es un gradiente: hay un salto de 0.60 entre el grupo alto y el medio, y el grupo bajo es
cero exacto.

In [ ]:
ALTO  = ["#1 Pick", "Best Seller", "Customer Favorite", "Top Rated"]
MEDIO = ["Highly Rated", "Popular Choice", "Shopper Favorite", "Well Reviewed"]

df["nivel"] = np.select(
    [df["sufijo"].isin(ALTO), df["sufijo"].isin(MEDIO)],
    ["ALTO", "MEDIO"],
    default="CERO",
)

niveles = (df.groupby("nivel", observed=True)["bought"]
             .agg(filas="size", compras="sum", tasa="mean")
             .reindex(["ALTO", "MEDIO", "CERO"]))
niveles["% de todas las compras"] = (niveles["compras"] / df["bought"].sum() * 100).round(1)
niveles

**El hallazgo central del trabajo.** El nivel ALTO son 1.931 filas (19% del dataset) y
concentra 1.249 de las 1.301 compras: el **96%**. El nivel CERO, 6.096 filas, tiene tasa de
compra **exactamente cero**.

Esto explica por qué una arquitectura tabular pura tiene techo en ROC 0.58 y por qué un
TF-IDF del título llega a 0.956: **la señal es un marcador de reputación insertado en el
texto**, no una interacción entre atributos del producto. El Transformer tiene que ser un
encoder de texto (decisión D5).

También explica por qué el sufijo **se mantiene** en el input (decisión D7). Sin él el
problema es irresoluble — el baseline de LogReg sobre el título sin sufijo da ROC 0.4991,
que es azar. El sufijo se **ablaciona** en la Fase 9, no se elimina.

### 7.2 La última oración de `description` codifica lo mismo

La descripción termina con una frase sobre la reputación del producto. Si esa frase
determina el nivel, entonces `description` es redundante con el sufijo del título — y eso
predice que la ablación "solo título / solo descripción / ambos" no va a mostrar diferencia.

In [ ]:
# pandas trata el patron de str.split como regex; se usa re para partir por oracion.
SEPARADOR_ORACION = re.compile(r"(?<=\.)\s+")
df["oracion_final"] = df["description"].map(lambda t: SEPARADOR_ORACION.split(t.strip())[-1].strip())

print(f"oraciones finales distintas: {df['oracion_final'].nunique()}")

cruce = pd.crosstab(df["oracion_final"], df["nivel"])[["ALTO", "MEDIO", "CERO"]]
niveles_por_oracion = (cruce > 0).sum(axis=1)
print(f"oraciones finales que aparecen en MÁS de un nivel: {(niveles_por_oracion > 1).sum()}")
cruce.loc[cruce.sum(axis=1).sort_values(ascending=False).index].head(15)

**Ninguna oración final aparece en más de un nivel.** El mapeo oración final → nivel es
determinístico: las 36 oraciones particionan limpiamente en los tres niveles.

Conviene ser preciso sobre la forma de la redundancia, porque el plan la enunciaba al revés:
el mapeo **sufijo → oración** *no* es uno a uno. Cada sufijo se reparte entre las 4
oraciones de su nivel, con ~27% cada una:

In [ ]:
(df.groupby("sufijo", observed=True)
   .agg(nivel=("nivel", "first"),
        oraciones_finales_distintas=("oracion_final", "nunique"),
        cobertura_de_la_mas_frecuente=("oracion_final",
                                       lambda s: round((s == s.mode().iloc[0]).mean(), 3)))
   .sort_values(["nivel", "sufijo"]))

Es decir: **el sufijo y la oración final son dos codificaciones independientes del mismo
nivel latente**, no copias una de la otra. Cada una alcanza por sí sola para recuperar el
nivel, que es lo que importa.

Los sufijos con ~20 oraciones distintas (`Current Stock`, `Standard Listing`,
`Regular Listing`, *sin sufijo*) son los que a veces **no traen oración de reputación**: su
descripción termina en la frase genérica "Listed under X and intended for Y storage". Son
459 filas, todas de nivel CERO.

### 7.3 El texto es plantillado

Si el texto contiene los mismos valores que las columnas tabulares, la rama de texto y la
rama tabular comparten información, y la fusión aporta menos de lo que parece.

In [ ]:
contenido = pd.DataFrame([
    {"verificación": "title contiene brand",
     "se cumple en": f"{np.mean([b in t for b, t in zip(df['brand'], df['title'])]):.2%}"},
    {"verificación": "description contiene package_size",
     "se cumple en": f"{np.mean([p in d for p, d in zip(df['package_size'], df['description'])]):.2%}"},
    {"verificación": "description contiene category",
     "se cumple en": f"{np.mean([c.lower() in d.lower() for c, d in zip(df['category'], df['description'])]):.2%}"},
])
contenido

Los tres al 100%. El texto es generado por plantilla a partir de las columnas tabulares más
el marcador de reputación. **La parte descriptiva no aporta información nueva**: lo único
que el texto tiene y las tablas no es el nivel de reputación.

Esto acota qué puede aportar la fusión texto + tabular de la Fase 8: el margen no está en
"más información", sino en que la rama tabular modela mejor los efectos de `allergens` y
`category` *dentro* del nivel (§9).

### 7.4 Longitud del texto — para dimensionar `max_len`

In [ ]:
longitudes = pd.DataFrame({
    "title": df["title"].str.split().str.len(),
    "description": df["description"].str.split().str.len(),
})
longitudes["title + description"] = longitudes["title"] + longitudes["description"]

resumen_long = longitudes.describe().T[["min", "mean", "max"]]
resumen_long["p95"] = longitudes.quantile(0.95)
resumen_long["p99"] = longitudes.quantile(0.99)
resumen_long.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(longitudes["title"], bins=range(4, 18), alpha=0.75, color=AZUL, label="title")
ax.hist(longitudes["description"], bins=range(17, 36), alpha=0.75, color=VERDE, label="description")
ax.set_xlabel("palabras"); ax.set_ylabel("filas")
ax.set_title("Longitud del texto en palabras")
ax.legend(fontsize=8)
guardar(fig, "fig_05_longitud_texto")
plt.show()

Títulos de 6 a 14 palabras, descripciones de 19 a 32, concatenación con p99 = 43 palabras.

**Consecuencia para la Fase 6:** con BPE sobre un corpus plantillado se esperan del orden de
80–110 tokens. Hay que **medirlo** sobre train y fijar `max_len` en el percentil 99, no
poner 512 por costumbre: con secuencias de ~90 tokens, 512 multiplica por más de 30 el costo
cuadrático de la atención sin ganancia alguna.

---
## 8. Bivariado marginal con el target

Lo que se ve mirando cada variable contra `bought` sin condicionar por nada.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 6))
columnas_biv = ["category", "brand", "storage_type", "country_of_origin", "unit_of_measure", "allergens"]

for ax, col in zip(axes.ravel(), columnas_biv):
    serie = df[col].fillna("[NONE]") if col == "allergens" else df[col]
    t = df.assign(_c=serie).groupby("_c", observed=True)["bought"].mean().sort_values()
    ax.barh(t.index.astype(str), t.values, color=AZUL)
    ax.axvline(df["bought"].mean(), color=ROJO, ls="--", lw=1.2)
    ax.set_title(f"{col}   (rango {t.min():.3f} – {t.max():.3f})", fontsize=9)
    ax.tick_params(labelsize=7)

fig.suptitle("Tasa de compra marginal por categórica (línea roja = prevalencia global 0.1301)",
             y=1.01, fontsize=11)
fig.tight_layout()
guardar(fig, "fig_06_bivariado_marginal")
plt.show()

In [ ]:
# Numericas por decil.
filas = []
for col in NUMERICAS + ["price_pos"]:
    deciles = pd.qcut(df[col], 10, labels=False, duplicates="drop")
    t = df.groupby(deciles, observed=True)["bought"].mean()
    filas.append({
        "variable": col,
        "corr con bought": round(df[col].corr(df["bought"].astype(int)), 4),
        "tasa decil 1": round(t.iloc[0], 4),
        "tasa decil 10": round(t.iloc[-1], 4),
        "rango entre deciles": round(t.max() - t.min(), 4),
    })
pd.DataFrame(filas)

**Cómo NO leer esta sección.** La correlación de `price` con `bought` es 0.003. Reportarla
como evidencia de algo sería un error. Lo mismo vale para `category`: el rango marginal
0.057–0.191 parece un efecto, pero está mayormente **confundido con el nivel de reputación**
— basta con que una categoría tenga más productos de nivel ALTO para que su tasa marginal
suba, sin que la categoría cause nada.

El único modo honesto de leer estas variables es condicionando por el nivel. Eso es la §9.

---
## 9. Análisis condicionado al nivel ALTO

La estructura del dataset es **jerárquica**: primero el nivel de reputación decide si la
compra es posible (en CERO es imposible), después los atributos del producto modulan la
probabilidad *dentro* del nivel. Un análisis marginal no lo muestra; este sí.

In [ ]:
alto = df[df["nivel"] == "ALTO"].copy()
print(f"filas de nivel ALTO : {len(alto):,}")
print(f"tasa base del nivel : {alto['bought'].mean():.4f}")

In [ ]:
alergenos_alto = tasa_por(alto.assign(allergens=alto["allergens"].fillna("[NONE]")), "allergens")
categoria_alto = tasa_por(alto, "category")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
base = alto["bought"].mean()

for ax, tabla, titulo in [(ax1, alergenos_alto, "allergens"), (ax2, categoria_alto, "category")]:
    t = tabla.sort_values("tasa")
    colores = [ROJO if v < base - 0.15 else AZUL for v in t["tasa"]]
    barras = ax.barh(t.index.astype(str), t["tasa"], color=colores)
    ax.axvline(base, color="black", ls="--", lw=1.4, label=f"tasa del nivel ALTO = {base:.3f}")
    ax.bar_label(barras, labels=[f"{v:.3f}  (n={int(n)})" for v, n in zip(t['tasa'], t['filas'])],
                 fontsize=7, padding=2)
    ax.set_xlim(0, 1.05)
    ax.set_title(f"Tasa de compra por {titulo}, DENTRO del nivel ALTO")
    ax.legend(fontsize=8, loc="lower right")

fig.tight_layout()
guardar(fig, "fig_07_condicionado_alto")
plt.show()

**Acá aparecen los efectos reales, y son grandes.**

Los alérgenos de origen marino y los frutos secos deprimen la compra a la mitad: Fish 0.291,
Shellfish 0.314, Tree nuts 0.337, Peanuts 0.394, contra Wheat 0.688, sin alérgeno 0.704,
Soy 0.705 y Milk 0.721. En `category`, Seafood 0.302 contra Bakery 0.742.

Estos efectos estaban **diluidos** en el análisis marginal, porque el nivel domina la
varianza. Son la razón por la que la fusión texto + tabular de la Fase 8 no es decorativa:
el baseline que combina sufijo y tabulares llega a PR-AUC 0.775, muy por encima de
cualquiera de las dos ramas por separado.

In [ ]:
print("Correlaciones DENTRO del nivel ALTO (contra la marginal sobre todo el dataset):\n")
for col in ["price_pos", "price", "net_weight_oz", "nutrition_score"]:
    r_alto = alto[col].corr(alto["bought"].astype(int))
    r_global = df[col].corr(df["bought"].astype(int))
    print(f"  {col:<18} dentro de ALTO {r_alto:+.4f}    marginal {r_global:+.4f}")

`price_pos` es la única numérica con efecto: 0.105 dentro del nivel ALTO, contra 0.039
marginal. Productos más caros dentro del rango del filtro se compran algo más. `price`,
`net_weight_oz` y `nutrition_score` siguen sin efecto (|r| < 0.04) incluso condicionando:
son ruido sintético.

In [ ]:
# ¿Compite el nivel ALTO consigo mismo dentro de una query?
n_alto_por_query = df[df["nivel"] == "ALTO"].groupby(L.GROUP).size().rename("items_alto")
competencia = (alto.join(n_alto_por_query, on=L.GROUP)
                   .groupby("items_alto", observed=True)["bought"]
                   .agg(filas="size", tasa="mean"))
competencia

**Resultado negativo, y es relevante.** La tasa de compra de un ítem de nivel ALTO es
prácticamente la misma haya 1, 2, 3 o 4 ítems ALTO compitiendo en la misma query
(0.643 / 0.645 / 0.669 / 0.630). El único grupo que se aparta tiene 5 filas: es ruido.

Las impresiones de una query **no compiten entre sí** por la compra. Esto anticipa que la
atención *cross-item* de la decisión A4 va a dar ganancia nula. Se implementa igual y se
reporta como resultado negativo, que es tan válido como uno positivo.

### 9.1 ¿Hay efecto temporal? — contra la banda de ruido

Antes de derivar features cíclicas de `timestamp`, hay que comprobar que la variación por
hora / día / mes supera lo que produciría el azar. Con `n` filas por grupo y tasa `p`, el
desvío esperado de las tasas por puro muestreo es `sqrt(p(1-p)/n)`.

In [ ]:
filas = []
for nombre, valores in [("hora", alto["timestamp"].dt.hour),
                        ("día de semana", alto["timestamp"].dt.dayofweek),
                        ("mes", alto["timestamp"].dt.month)]:
    g = alto.groupby(valores, observed=True)["bought"].agg(["size", "mean"])
    n_medio = g["size"].mean()
    p = alto["bought"].mean()
    ruido = np.sqrt(p * (1 - p) / n_medio)
    filas.append({
        "variable": nombre,
        "grupos": len(g),
        "n medio por grupo": int(n_medio),
        "desvío observado": round(g["mean"].std(), 4),
        "desvío esperado por azar": round(ruido, 4),
        "razón obs/azar": round(g["mean"].std() / ruido, 2),
    })
pd.DataFrame(filas)

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3.2))
por_hora = alto.groupby(alto["timestamp"].dt.hour, observed=True)["bought"].agg(["size", "mean"])
p = alto["bought"].mean()
banda = 1.96 * np.sqrt(p * (1 - p) / por_hora["size"])

ax.errorbar(por_hora.index, por_hora["mean"], yerr=banda, fmt="o", color=AZUL,
            capsize=3, markersize=4, lw=1)
ax.axhline(p, color=ROJO, ls="--", lw=1.4, label=f"tasa del nivel ALTO = {p:.3f}")
ax.set_xlabel("hora UTC"); ax.set_ylabel("tasa de compra")
ax.set_title("Tasa de compra por hora, dentro del nivel ALTO (IC 95%)")
ax.legend(fontsize=8)
guardar(fig, "fig_08_temporal_vs_ruido")
plt.show()

El desvío observado está en el orden del esperado por azar (razón ≈ 1) y todos los intervalos
de confianza cruzan la tasa base. **No hay efecto temporal**: las features cíclicas de
`timestamp` se descartan y el descarte queda documentado, que es lo que pide la Fase 4.

---
## 10. Confusión Seafood / Fish-Shellfish

`category` y `allergens` no son independientes. Antes de atribuirle un efecto a cada una,
hay que ver cuánto se solapan.

In [ ]:
cruce_cat_alerg = pd.crosstab(df["category"], df["allergens"].fillna("[NONE]"))
cruce_cat_alerg

In [ ]:
seafood = df[df["category"] == "Seafood"]
marinos = df[df["allergens"].isin(["Fish", "Shellfish"])]
print(f"P(allergens ∈ {{Fish, Shellfish}} | category = Seafood) = {seafood['allergens'].isin(['Fish','Shellfish']).mean():.4f}")
print(f"P(category = Seafood | allergens ∈ {{Fish, Shellfish}}) = {(marinos['category'] == 'Seafood').mean():.4f}")

**Son la misma señal.** Fish y Shellfish aparecen *solo* en Seafood, y todo Seafood tiene
uno de los dos. La correspondencia es perfecta en ambos sentidos.

Consecuencia práctica: no se puede separar el efecto "es pescado" del efecto "tiene alérgeno
marino", porque son la misma partición de las filas. Incluir las dos columnas duplica la
señal y hace ininterpretable cualquier medida de importancia de features. Se reporta como
una limitación y la ablación de la Fase 9 (`con / sin allergens`, `con / sin category`) se
lee teniéndolo presente.

El resto de la tabla muestra el mismo patrón más suave: Dairy → Milk, Bakery → Wheat.
`allergens` es en buena medida una función de `category`.

---
## 11. Las cuatro respuestas del Ejercicio 1

### 11.1 ¿Qué se predice?

Se predice el **Buy Through Rate**: la probabilidad de que una impresión de producto
termine en compra, dentro del contexto de la búsqueda que la generó.

- **Unidad de observación:** la impresión, es decir el par (producto, query). 10.000 filas.
- **Target:** `bought`, binario. Prevalencia **0.1301** (1.301 positivos).
- **Formulación:** clasificación binaria a nivel impresión. La probabilidad predicha *es*
  el BTR, sin transformación adicional.
- **Momento de la predicción:** el instante previo a mostrar la página de resultados, cuando
  el sistema decide qué productos promocionar.
- **Alternativa descartada:** una formulación *listwise* con softmax sobre la query, que
  asumiría una compra por búsqueda. Se descarta porque 284 queries tienen más de una compra
  (§4).
- **Métricas:** PR-AUC (*average precision*) como principal, porque con 13% de positivos el
  ROC-AUC es optimista; ROC-AUC como secundaria; log loss y Brier para calibración, que
  importan porque el output es una probabilidad con significado propio.

### 11.2 ¿Qué características tiene la información?

**Estructura.** 10.000 impresiones × 22 columnas, agrupadas en 2.012 queries de 1 a 8 ítems
(media 4.97). 8.578 productos únicos. Sin duplicados exactos; un solo duplicado
`(query_id, title)`.

**Desbalance.** 13% de positivos. Moderado: no requiere remuestreo, pero sí que la métrica
principal sea PR-AUC y que el bias de salida se inicialice en `log(0.13/0.87)`.

**Nulos.** Solo en `allergens` (44.55%), y son **informativos**: su tasa de compra (0.1385)
cae entre la de Milk y la de Soy. Se codifican como categoría propia.

**Jerarquía.** El dataset tiene dos niveles. Un marcador de reputación en el texto decide si
la compra es *posible* (en el nivel CERO, 6.096 filas, la tasa es exactamente 0); dentro del
nivel ALTO, los atributos del producto modulan la probabilidad. Cualquier análisis que no
condicione por el nivel confunde los dos efectos.

**Tiempo.** El `timestamp` cubre 730 días pero **no ordena las queries**: el rango dentro de
una misma query tiene mediana de 488 días. No hay estructura temporal explotable, y no
corresponde un split temporal.

**Texto plantillado.** `title` y `description` se generan por plantilla desde las columnas
tabulares. `title` contiene `brand` en el 100% de las filas; `description` contiene
`package_size` y `category` en el 100%. Lo único que el texto aporta y las tablas no es el
marcador de reputación.

**Dataset sintético.** La señal es un token categórico insertado en el texto, no una
interacción semántica. Es honesto decirlo: acota lo que cualquier modelo puede aprender acá,
y explica por qué se espera que un GBDT con el sufijo como categórica iguale o supere al
Transformer.

### 11.3 ¿Qué features se usan?

De las 22 columnas: **17 admitidas**, 3 excluidas, 1 target, 1 clave de agrupamiento.

| Grupo | Columnas |
|---|---|
| Texto | `title`, `description` |
| Numéricas | `price`, `net_weight_oz`, `nutrition_score`, `filter_price_min`, `filter_price_max` |
| Categóricas | `category`, `brand`, `storage_type`, `country_of_origin`, `unit_of_measure`, `package_size`, `ingredients`, `allergens` |
| Derivadas | `price_pos`, `price_per_oz`, largo/ancho/alto/volumen desde `dimensions_in` |
| Descartadas tras el EDA | features cíclicas de `timestamp` (§9.1: la variación no supera el ruido) |

**Excluidas, con motivo:**

| Columna | Motivo |
|---|---|
| `cart` | **Fuga.** `P(bought=1 \| cart=False) = 0` exacto. Su valor es posterior al momento de la predicción. |
| `filter_category` | Idéntica a `category` en el 100% de las filas. |
| `filter_storage_type` | Idéntica a `storage_type` en el 100% de las filas. |

`query_id` es la clave de agrupamiento de la partición, no una feature: 2.012 valores únicos,
codificarla solo memoriza. Tampoco se construyen features de coherencia con el filtro,
porque las tres condiciones se cumplen al 100% y serían constantes (§6).

### 11.4 ¿Qué preprocesamiento tiene cada una?

| Tipo | Preprocesamiento | Por qué |
|---|---|---|
| `title`, `description` | `[CLS] title [SEP] description [SEP]`, BPE entrenado **solo con train**, vocab 2k–4k, `max_len` en el p99 medido | Vocabulario chico porque el corpus es plantillado y todos los embeddings se aprenden desde cero; un vocab de 30k dejaría casi todos sin entrenar |
| Sufijo del título | **Se mantiene** en el input | Sin él el problema es irresoluble (ROC 0.499). Se ablaciona en la Fase 9, no se elimina |
| Numéricas asimétricas (`price`, `net_weight_oz`, `filter_price_min`) | `QuantileTransformer` o `log1p` + `StandardScaler`, ajustado **solo con train** | Skew de 1.5 a 2.8 (§2) |
| Numéricas simétricas | `StandardScaler`, ajustado solo con train | |
| Derivadas de `dimensions_in` | Regex sobre los tres floats de `"3.3 x 4.0 x 4.1\""`; parsea el 100% de las filas | |
| Categóricas | One-hot para los baselines; `nn.Embedding` de dim 8–16 por columna para el Transformer | Cardinalidad baja o media: sin bucket `[RARE]` (§3) |
| `allergens` | Categoría propia `"[NONE]"` para el nulo, **sin imputar** | El nulo es informativo (§3) |
| `ingredients` | Categórica; candidata a descarte | 190 valores, pero mediana de 3 filas por valor dentro del nivel ALTO: demasiado ralo para estimar un efecto |
| `timestamp` | Descartada | §9.1 |

**Regla transversal:** todo transformador con estado (escalador, one-hot, vocabulario BPE)
se ajusta **solo con train**, por fold. Ajustarlo sobre el dataset completo mete la media
del test en la normalización de train: es la fuga más frecuente y la más fácil de evitar con
disciplina de pipeline.

---

## Resumen de decisiones que este EDA justifica

| Decisión | Evidencia |
|---|---|
| D1 · Clasificación binaria a nivel impresión, no *listwise* | 284 queries con más de una compra (§4) |
| D2 · Excluir `cart` | `P(bought \| cart=False) = 0` exacto y posterior al momento de la predicción (§0) |
| D3 · Partición agrupada por `query_id` | Las filas de una query comparten filtros y contexto (§4) |
| D4 · **No** split temporal | Rango intra-query con mediana de 488 días sobre 730 (§5) |
| D5 · El Transformer opera sobre texto | La señal es un marcador textual; el nivel CERO tiene tasa 0 (§7.1) |
| D6 · No construir features de match con filtros | Las tres condiciones se cumplen al 100% (§6) |
| D7 · Mantener el sufijo en el input | Sin él, ROC 0.499 (§7.1) |
| D8 · PR-AUC como métrica principal | Prevalencia 0.1301 (§11.1) |
| A4 · Atención *cross-item* como ablación de baja prioridad | La tasa no depende de cuántos ítems ALTO compitan (§9) |